# Conexión a la BD SQL 2025

In [1]:
import pyodbc

# Configuración de conexión según tus capturas previas
# Cambia 'TU_CONTRASEÑA' por la que asignaste al usuario CPFSQL2025
config = {
    'driver': '{ODBC Driver 17 for SQL Server}', # Usando el driver de tu captura
    'server': 'JOEL-LT',                         # Tu servidor
    'database': 'NDFPCYFCH',                     # Tu BD de Fiscalía
    'user': 'CPFSQL2025',                        # Tu usuario creado
    'password': 'A1JOEL*@#'                  
}

def probar_conexion():
    try:
        conn_str = (
            f"DRIVER={config['driver']};"
            f"SERVER={config['server']};"
            f"DATABASE={config['database']};"
            f"UID={config['user']};"
            f"PWD={config['password']};"
            "Encrypt=no;" # Recomendado para conexiones locales de prueba
        )
        
        conn = pyodbc.connect(conn_str)
        print("✅ ¡Éxito! Python se conectó a la BD de la Fiscalía de Familia.")
        
        # Una pequeña consulta para validar acceso a tablas
        cursor = conn.cursor()
        cursor.execute("SELECT TOP 1 name FROM sys.tables")
        row = cursor.fetchone()
        if row:
            print(f"🔎 Conexión verificada. Primera tabla encontrada: {row[0]}")
            
        conn.close()
    except Exception as e:
        print(f"❌ Error al conectar: {e}")

if __name__ == "__main__":
    probar_conexion()

✅ ¡Éxito! Python se conectó a la BD de la Fiscalía de Familia.
🔎 Conexión verificada. Primera tabla encontrada: CARGOS


In [2]:
import pyodbc
import os

# Configuración de rutas
output_dir = '../documentos'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=JOEL-LT;"
    "DATABASE=NDFPCYFCH;"
    "UID=CPFSQL2025;"
    "PWD=A1JOEL*@#;"
)

# Consulta SQL con todos los campos para el nombre del archivo
query = """
SELECT TOP 10
    L.[DOCUMENTO], 
    L.[EXT_DOC_CASO], 
    N.[DESPACHO],
    N.[COD_NUM_DOC],
    C.[AÑOCASO],
    C.[NUMCASO],
    C.[SECCUENCIACASO],
    N.[TIPO_DOCUMENTO]
FROM [NDFPCYFCH].[dbo].[LEGADOCCASO] L
INNER JOIN [NDFPCYFCH].[dbo].[NUMERODOC] N ON L.DESPACHO = N.DESPACHO AND L.COD_NUM_DOC = N.COD_NUM_DOC
INNER JOIN [NDFPCYFCH].[dbo].[CASO] C ON N.DESPACHO = C.CODDESPACHO AND N.COD_NUM_DOC = C.CODNUMDOC
WHERE N.TIPO_DOCUMENTO = 2 
  AND N.ESTADODOC = 3 
  AND N.FECHA >= '2026-01-01'
"""

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    cursor.execute(query)
    
    rows = cursor.fetchall()
    print(f"📂 Se encontraron {len(rows)} registros. Iniciando extracción...")

    for row in rows:
        blob_data = row[0]
        # Aseguramos que la extensión tenga un punto al inicio
        ext_raw = str(row[1]).strip() if row[1] else ".docx"
        extension = ext_raw if ext_raw.startswith('.') else f".{ext_raw}"
        
        despacho  = str(row[2]).strip()
        cod_num   = str(row[3]).strip()
        anio      = str(row[4]).strip()
        num_caso  = str(row[5]).strip()
        secuencia = str(row[6]).strip()
        tipo_doc  = str(row[7]).strip()

        # Construcción del nombre con el PUNTO para la extensión
        # Formato: DESPACHO-COD_NUM_DOC-AÑOCASO-NUMCASO-SECCUENCIACASO-TIPO_DOCUMENTO.ext
        nombre_archivo = f"{despacho}-{cod_num}-{anio}-{num_caso}-{secuencia}-{tipo_doc}{extension}"
        
        filepath = os.path.join(output_dir, nombre_archivo)

        if blob_data:
            with open(filepath, 'wb') as f:
                f.write(blob_data)
            print(f"✅ Generado correctamente: {nombre_archivo}")

    conn.close()
    print("\n🚀 Proceso completado con éxito.")

except Exception as e:
    print(f"❌ Error durante la extracción: {e}")

📂 Se encontraron 10 registros. Iniciando extracción...
✅ Generado correctamente: 6411-53972-2026-1-0-2.doc
✅ Generado correctamente: 6411-53981-2026-2-0-2.doc
✅ Generado correctamente: 6411-54017-2026-5-0-2.doc
✅ Generado correctamente: 6411-54052-2026-6-0-2.doc
✅ Generado correctamente: 6411-54146-2026-17-0-2.doc
✅ Generado correctamente: 6411-54281-2026-33-0-2.doc
✅ Generado correctamente: 6411-54320-2026-38-0-2.doc
✅ Generado correctamente: 6411-54356-2026-42-0-2.doc
✅ Generado correctamente: 6411-54362-2026-40-0-2.doc
✅ Generado correctamente: 6411-54368-2026-41-0-2.doc

🚀 Proceso completado con éxito.
